In [1]:
import json, os

In [2]:
input_folder = "/home/lamdo/IllinoisRetrievalBenchmark/test_run_gitig_10"

step1_files = os.listdir(os.path.join(input_folder, "step1"))

In [3]:
test_data = []
from collections import Counter
counter = Counter()
for file in step1_files:
    step1 = os.path.join(input_folder, "step1", file)
    step2_1 = os.path.join(input_folder, "step2_1", file)
    step2_2 = os.path.join(input_folder, "step2_2", file)
    step3 = os.path.join(input_folder, "step3", file)

    try:

        with open(step1) as f:
            step1_data = json.load(f)

        with open(step2_1) as f:
            step2_1_data = json.load(f)

        with open(step2_2) as f:
            step2_2_data = json.load(f)
        
        with open(step3) as f:
            step3_data = json.load(f)
    except FileNotFoundError: continue

    url_content_mapper = step2_1_data.get("url_content_mapper")
    keypoints_mapper = step2_2_data.get("keypoints_mapper")
    groundedness_check = step3_data.get("groundedness_check")
    if not groundedness_check or not keypoints_mapper: continue

    visited = set()
    for k, score in groundedness_check.items():
        fact_id, url, kp_id = k.split("--__--")
        fact_id = int(fact_id)
        kp_id = int(kp_id)


        if fact_id not in visited:
            counter[score] += 1
            visited.add(fact_id)
        if score: continue


        keypoint = keypoints_mapper[str(fact_id)][kp_id]

        url_content = url_content_mapper[url]["url_content"]
        published_date = url_content_mapper[url]["published_date"]
        lang = url_content_mapper[url]["lang"]

        if not lang or lang == "en": continue

        test_data.append(["Fact: "+ keypoint, "Context Published date: " + str(published_date), "Context Language: " + str(lang), "Context: "+ url_content])


In [4]:
counter

Counter({False: 970, True: 2202})

In [9]:
import gzip

In [11]:
with gzip.open("/scratch/lamdo/wiki_dump/enwiki-20250630-cirrussearch-content.json.gz", 'rt', encoding='utf-8') as f:
    test_data = []
    for idx, line in enumerate(f):
        obj = json.loads(line) 
        if isinstance(obj, dict) and obj.get("source_text") is not None:
            test_data.append(obj)

        if len(test_data) == 100: break

In [14]:
test_data[2]

{'template': ['Template:More citations needed',
  'Template:Ambox',
  'Template:Find sources mainspace',
  'Template:Infobox film',
  'Template:Main other',
  'Template:Infobox film/short description',
  'Template:Has short description',
  'Template:Short description',
  'Template:Film date',
  'Template:Plainlist',
  'Template:Plainlist/styles.css',
  'Template:In string',
  'Template:Start date',
  'Template:Pagetype',
  'Template:Short description/lowercasecheck',
  'Template:SDcat',
  'Template:Infobox',
  'Template:If empty',
  'Template:Longitem',
  'Template:Pluralize from text',
  'Template:Template other',
  'Template:Str find',
  'Template:Str count',
  'Template:Empty section',
  'Template:IMDB title',
  'Template:IMDb title',
  'Template:Wikidata',
  'Template:Trim',
  'Template:EditAtWikidata',
  'Template:WikidataCheck',
  'Template:1970s-Argentina-film-stub',
  'Template:Asbox',
  'Template:Article stub box',
  'Template:Hlist/styles.css',
  'Module:Unsubst',
  'Module:M